# 01 — Exploración ECoG y QC de adquisición
Selecciona el kernel **Colab** en VS Code. Este notebook solo analiza desarrollo de S1/S2/S3; no limpia señales, entrena ni consulta prueba.

El código debe estar publicado en el remoto antes de clonar. Si el repositorio es privado, autentica Git en la sesión mediante un mecanismo interactivo; no escribas tokens en celdas ni URLs. Las salidas del notebook pueden contener información privada: no publicarlas.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
PROJECT = Path('/content/tt-ecog')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa la carpeta.')
print('Commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())
print('Se conserva el checkout existente. Para actualizarlo, revisa cambios y usa git pull --ff-only explícitamente.')

## Entorno de EDA
Se inspeccionan versiones y se instalan solamente dependencias EDA ausentes. Una versión incompatible detiene el arranque con diagnóstico. No se instala PyTorch ni CUDA. Si modificas los módulos después de importarlos, reinicia el kernel para que el código ejecutado coincida con los hashes.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts/check_eda_environment.py'), '--install-missing'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)

## Drive y configuración única
Puedes usar `Colab: Mount Google Drive to Server...` o ejecutar la celda siguiente. Las rutas relativas se leen del YAML: datos en `TT_ECoG/raw` y resultados en `TT_ECoG/results`. No se buscan ni resuelven etiquetas de prueba.

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
from tt_ecog.config import load_config
CONFIG = PROJECT / 'configs/experiment_config.yaml'
config = load_config(CONFIG)
print('Datos de desarrollo:', config.path(DRIVE_ROOT, 'raw_dir'))
print('Resultados:', config.path(DRIVE_ROOT, 'results_dir'))
print('SHA-256 del YAML:', config.sha256)

## Ejecución de S1/S2/S3
La primera ejecución produce estadísticas y figuras sin rellenar umbrales `null`. Los detectores correspondientes se identifican como `PENDING`. Después de revisar esta evidencia, registra candidatos en el YAML, actualiza el checkout y vuelve a ejecutar. Los candidatos no autorizan exclusiones.

Cada ejecución tiene una carpeta nueva y un manifiesto de hashes. Los datos ausentes producen `BLOCKED`; una integridad inválida produce `FAIL`. Ninguno equivale a aprobación.

In [ ]:
from tt_ecog.eda import run_eda
import json
import pandas as pd
from IPython.display import display
RUN = run_eda(CONFIG, DRIVE_ROOT)
summary = json.loads((RUN / 'run_summary.json').read_text(encoding='utf-8'))
print('Estado:', summary['status'])
print('Evidencia persistida en:', RUN)
display(pd.read_csv(RUN / 'qc_summary.csv'))
if summary['status'] in {'BLOCKED', 'FAIL'}:
    raise RuntimeError('EDA incompleto. Consulta qc_summary.csv y los diagnósticos antes de continuar.')

In [ ]:
from IPython.display import Image, Markdown
display(Markdown((RUN / 'qc_review_log.md').read_text(encoding='utf-8')))
display(pd.read_csv(RUN / 'qc_channels.csv'))
display(pd.read_csv(RUN / 'qc_events.csv'))
for subject in config.data['project']['subjects']:
    print(subject, json.loads((RUN / f'{subject}_detectors.json').read_text(encoding='utf-8')))
    for figure in sorted((RUN / 'qc_figures').glob(f'{subject}_*.png')):
        display(Image(filename=str(figure)))

## Revisión humana pendiente
`qc_decisions.json` permanece `pending`. Revisa los tres sujetos, fundamenta umbrales y documenta cada acción con índices, tiempos, razones y evidencia. La aprobación requiere revisor y fecha; este notebook no modifica estados de aprobación.

Esta entrega no implementa preprocesamiento ni las verificaciones V01–V14 completas. No iniciar etapas posteriores hasta aprobar QC. Conserva la carpeta de ejecución y sus hashes; no publiques las salidas del notebook en Git.